In [ ]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数

    评测时平台会自动替换 datasources / start_date / end_date 三个入参并调用本函数

    参数:
        datasources (dict): 数据源表名映射 {逻辑名: 物理表名}。一个因子可同时用到多张表，
                            通过逻辑名取出该阶段实际的物理表名，平台会在公榜/私榜自动切换。
                            当前可用逻辑名:
                                "bar1m"     -> 分钟 K 线表
                                "financial" -> 财务数据表
        start_date (str): 开始时间
        end_date (str):   结束时间

    返回:
        pd.DataFrame: 因子数据，须包含三列 ['date', 'instrument', 'factor']，且不含 inf
    """
    import pandas as pd
    import dai

    # 从映射里取出本阶段实际的物理表名（切勿在 SQL 里硬编码表名，否则公榜/私榜无法切换）
    bar1m = datasources["bar1m"]

    # 若计算滚动/时序类因子，可以多取若干天数据作为缓冲（本示例无需滚动，仅作演示）
    LOOKBACK_DAYS = 7
    query_start_date = pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)

    # ===== 编写因子 SQL =====
    # 示例因子：基于 1 分钟盘口快照计算日内订单簿压力，按交易日聚合为日频因子
    # DAI 函数文档：https://bigquant.com/wiki/doc/Rceb2JQBdS
    sql = f"""
    -- ============== late_amount_absorption_reversal ==============
    -- Economic intuition:
    --   Late-session concentration carries more information when price closes
    --   below its full-day amount-weighted mid-price.  The product measures
    --   absorption / reversal pressure; a larger value is more bullish.
    -- Exact pre-registered formula:
    --   vwap              = SUM(amount * mid_price) / SUM(amount)
    --   late_amount_share = SUM(amount after 14:30) / SUM(amount)
    --   raw               = ((vwap - close_mid_price) / vwap) * late_amount_share
    --   factor            = 2 * cross_sectional_percent_rank(raw) - 1
    WITH minute_base AS (
        SELECT
            date,
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            amount,
            (ask_price1 + bid_price1) / 2 AS mid_price
        FROM {bar1m}
        WHERE date >= '{start_date}' AND date < '{end_date}'
          AND ask_price1 > 0 AND bid_price1 > 0
          AND amount IS NOT NULL AND amount >= 0
    ),
    daily_raw AS (
        SELECT
            trading_day,
            instrument,
            SUM(amount * mid_price) / NULLIF(SUM(amount), 0) AS vwap,
            last(mid_price ORDER BY date) AS close_mid_price,
            SUM(
                CASE
                    WHEN strftime(date, '%H:%M:%S') >= '14:30:00'
                    THEN amount ELSE 0
                END
            ) / NULLIF(SUM(amount), 0) AS late_amount_share
        FROM minute_base
        GROUP BY trading_day, instrument
    ),
    daily_signal AS (
        SELECT
            trading_day,
            instrument,
            ((vwap - close_mid_price) / NULLIF(vwap, 0))
                * late_amount_share AS raw_factor
        FROM daily_raw
    )
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        2 * percent_rank() OVER (
            PARTITION BY trading_day ORDER BY raw_factor
        ) - 1 AS factor
    FROM daily_signal
    """
    # ===== 调用dai计算因子 =====
    # compression=True 会把 instrument 列转为 category 类型，显著降低内存占用
    df = dai.query(sql, filters={'date': [start_date, end_date]}, compression=True).df()

    # ===== 对齐股票池 =====
    # 数据源保留了 2019 年至今所有成分股的数据以便计算时序因子，
    # 因此需与中证 1000 成分股做内连接，只保留当日属于成分股的标的
    # bigalpha_2026_instruments 已经收录了2019年以来的所有数据，不用替换
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    df = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])

    return df


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时自行构造数据源映射（评测时由平台注入，逻辑名固定为 "bar1m"/"financial"）
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 读取平台因子库用于回归评估，您可以换成自己的因子库
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统：
    # process_pools=False 表示不对因子库再做预处理（bigalpha_2026_factorlib 已处理过）
    # show=True 表示画出评估图表
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
